In [1]:
import llb

In [5]:
import os

text = "I have a coin and observed flips. What is the true bias?"
data = {"flips": [0, 1, 0, 1, 1, 0, 1, 1, 0, 1]}
targets = ["true"]
#I have delete the API Key after this cell, so it is not valid anymore, please change it so that it can be run.

posterior = llb.infer(
    text=text,
    data=data,
    targets=targets,
    api_url="https://api.openai.com/v1/chat/completions",
    api_key="YOUR_OPENAI_API_KEY",
    api_model="gpt-4.1-mini",
    n_models=5,
)


Number of requested models: 5
Number of generated models: 5
Number of deduplicated models: 0
Number of invalid models (syntax/parsing): 0
Number of generation request failures (timeout/network/API): 0
Number of models missing required targets: 0
Number of models that failed to compile: 0
Number of models that failed during inference: 0
Number of models dropped due to target shape mismatch: 0
Number of models dropped due to non-finite log bound: 0
Number of valid models used in final aggregation: 5
--- Model Averaging Summary ---
Target: true
Number of models: 5
flat mean prediction: 0.589471
weighted mean prediction: 0.589696
Difference (weighted - flat): 0.000225

Top 5 models by weight for target 'true':
model=3, weight=0.212657, mu_i=0.588037
model=0, weight=0.212508, mu_i=0.578216
model=2, weight=0.209015, mu_i=0.590086
model=4, weight=0.208631, mu_i=0.606879
model=1, weight=0.157189, mu_i=0.584139
2 least-weighted models for target 'true':
model=1, weight=0.157189, mu_i=0.584139
m

In [ ]:
import csv
import io
import re
from contextlib import redirect_stdout
from datetime import datetime
from datetime import timedelta

import llb
import numpy as np

# Build USD test from real data (Close only): train on <= yesterday, predict today.
csv_path = "data.csv"

# Provided today's observed close.
today_date = datetime.strptime("03/20/2026", "%m/%d/%Y")
today_close = 99.65
yesterday_date = today_date - timedelta(days=1)

rows = []
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        dt = datetime.strptime(r["Date"], "%m/%d/%Y")
        close = float(str(r["Close"]).replace(",", "").replace('"', ""))
        rows.append((dt, close))

if len(rows) < 1:
    raise ValueError("CSV has no rows.")

rows.sort(key=lambda x: x[0])
train_rows = [(dt, v) for dt, v in rows if dt.date() <= yesterday_date.date()]
if len(train_rows) < 2:
    raise ValueError("Need at least 2 closes up to yesterday for this test.")

usd_index = [v for _, v in train_rows]
num_days = len(usd_index)

text = (
    "I have daily observations of the U.S. dollar index close values up to yesterday. "
    "Use only these close prices to model latent level and latent volatility, "
    "then predict today's close (one-day-ahead forecast from yesterday)."
)

data = {
    "num_days": num_days,
    "usd_index": usd_index,
}

targets = ["latent_level", "latent_volatility", "future_usd_index"]

# Run infer and capture its printed weighted-vs-flat summary so we can compare both to today.
buf = io.StringIO()
with redirect_stdout(buf):
    posterior_weighted = llb.infer(
        text=text,
        data=data,
        targets=targets,
        api_url="https://api.openai.com/v1/chat/completions",
        api_key="YOUR_OPENAI_API_KEY",
        api_model="gpt-4.1-mini",
        n_models=1,
        mcmc_num_warmup=500,
        mcmc_num_samples=1000,
        random_seed=42,
    )
infer_log = buf.getvalue()
print(infer_log, end="")
# Parse model-sharing diagnostics from infer logs for the USD forecast target.
target_name = "future_usd_index"
n_models_in_target = None
m_count = re.search(
    rf"--- Model Averaging Summary ---\nTarget: {re.escape(target_name)}\nNumber of models: (\d+)",
    infer_log,
    flags=re.MULTILINE,
 )
if m_count:
    n_models_in_target = int(m_count.group(1))

top_weights = []
m_top_block = re.search(
    rf"Top 5 models by weight for target '{re.escape(target_name)}':\n([\s\S]*?)\n2 least-weighted models for target '{re.escape(target_name)}':",
    infer_log,
    flags=re.MULTILINE,
 )
if m_top_block:
    top_weights = [
        float(x)
        for x in re.findall(r"weight=([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", m_top_block.group(1))
    ]

top1_weight = top_weights[0] if len(top_weights) >= 1 else None
top2_weight = top_weights[1] if len(top_weights) >= 2 else None
log_gap_top1_top2 = None
if top1_weight is not None and top2_weight is not None and top1_weight > 0 and top2_weight > 0:
    log_gap_top1_top2 = float(np.log(top1_weight / top2_weight))

n_eff_exact = None
n_eff_low = None
n_eff_high = None
if n_models_in_target is not None and len(top_weights) > 0:
    remaining_n = max(0, n_models_in_target - len(top_weights))
    remaining_mass = max(0.0, 1.0 - float(np.sum(top_weights)))
    sum_sq_top = float(np.sum(np.square(top_weights)))
    if remaining_n == 0 and abs(remaining_mass) < 1e-9:
        n_eff_exact = 1.0 / sum_sq_top if sum_sq_top > 0 else float("inf")
    else:
        sum_sq_high = sum_sq_top + (remaining_mass ** 2)
        sum_sq_low = sum_sq_top + ((remaining_mass ** 2) / remaining_n if remaining_n > 0 else 0.0)
        n_eff_low = 1.0 / sum_sq_high if sum_sq_high > 0 else float("inf")
        n_eff_high = 1.0 / sum_sq_low if sum_sq_low > 0 else float("inf")

print("--- Weight Concentration Diagnostics (future_usd_index) ---")
print("models in target summary:", n_models_in_target)
print("top-1 weight:", top1_weight)
print("top-2 weight:", top2_weight)
print("top-1/top-2 log gap:", log_gap_top1_top2)
if n_eff_exact is not None:
    print("N_eff (exact):", n_eff_exact)
elif n_eff_low is not None and n_eff_high is not None:
    print("N_eff lower bound (remaining mass concentrated):", n_eff_low)
    print("N_eff upper bound (remaining mass spread evenly):", n_eff_high)
else:
    print("N_eff: unavailable from current infer log format")

# Weighted mean from returned posterior.
fut_w = np.asarray(posterior_weighted["future_usd_index"], dtype=float)
pred_w_samples = fut_w if fut_w.ndim == 1 else fut_w[:, 0]
pred_w_mean = float(np.mean(pred_w_samples))

# Flat mean parsed from infer's "Weighted vs Flat Target Summary" block for future_usd_index.
flat_mean_value = None
block_pattern = r"--- Weighted vs Flat Target Summary ---\nTarget: future_usd_index\n([\s\S]*?)(?:\n---|\Z)"
m = re.search(block_pattern, infer_log)
if m:
    block = m.group(1)
    m_scalar = re.search(r"flat mean prediction:\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", block)
    if m_scalar:
        flat_mean_value = float(m_scalar.group(1))
    else:
        m_vec = re.search(r"flat mean prediction (?:full|first_10):\s*\[([^\]]+)\]", block)
        if m_vec:
            nums = [float(x) for x in re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", m_vec.group(1))]
            if nums:
                flat_mean_value = float(nums[0])

if flat_mean_value is None:
    raise RuntimeError("Could not parse flat mean prediction for future_usd_index from infer output.")

# Compare weighted and flat forecasts to today's known close.
err_w = pred_w_mean - today_close
err_f = flat_mean_value - today_close
abs_err_w = abs(err_w)
abs_err_f = abs(err_f)
pct_err_w = (abs_err_w / today_close) * 100.0
pct_err_f = (abs_err_f / today_close) * 100.0

print("Returned target keys (weighted):", list(posterior_weighted.keys()))
print(f"Train range: {train_rows[0][0].date()} to {train_rows[-1][0].date()} ({num_days} closes)")
print(f"Yesterday used as cutoff: {yesterday_date.date()}")
print(f"Predicted day (today): {today_date.date()}")
print(f"Actual close ({today_date.date()}): {today_close:.4f}")

print("--- Weighted vs Flat Compared To Today ---")
print(f"Weighted predicted mean: {pred_w_mean:.4f}")
print(f"Weighted error: {err_w:+.4f} (abs={abs_err_w:.4f}, pct={pct_err_w:.2f}%)")
print(f"Flat predicted mean: {flat_mean_value:.4f}")
print(f"Flat error: {err_f:+.4f} (abs={abs_err_f:.4f}, pct={pct_err_f:.2f}%)")

winner = "weighted" if abs_err_w < abs_err_f else ("flat" if abs_err_f < abs_err_w else "tie")
print(f"Better against today: {winner}")

Number of requested models: 200
Number of generated models: 199
Number of deduplicated models: 0
Number of invalid models (syntax/parsing): 1
Number of generation request failures (timeout/network/API): 0
Number of models missing required targets: 0
Number of models that failed to compile: 0
Number of models that failed during inference: 13
Number of models dropped due to target shape mismatch: 37
Number of models dropped due to non-finite log bound: 0
Number of valid models used in final aggregation: 149
--- Model Averaging Summary ---
Target: latent_level
Number of models: 149
flat mean prediction shape: (22,)
flat mean prediction first_10: [97.82140777157781, 97.87889658729506, 97.82265953852502, 97.74029565373361, 97.81071818409343, 97.75947295226031, 97.76961608313238, 97.81359720278746, 98.36180473634859, 98.86040008232574]
weighted mean prediction shape: (22,)
weighted mean prediction first_10: [97.86307584915373, 97.89002843474377, 97.80276232568032, 97.73327202868346, 97.80785